# 레이크하우스 접속 스타터

호스트 Jupyter에서 kind 클러스터의 Iceberg 레이크하우스에 붙는 최소 예제.

## 전제 — port-forward

웹 UI만 Ingress로 내고 **데이터 접속은 `port-forward`** 가 규약이다(`docs/conventions/k8s.md` §10).
노트북을 켜기 전에 별도 터미널에서:

```shell
kubectl port-forward svc/spark-connect 15002:15002       # 필수 (Spark SQL)
kubectl port-forward svc/catalog-postgres-rw 15432:5432  # 선택 (pyiceberg 직접 접속)
kubectl port-forward svc/seaweedfs 18333:8333            # 선택 (pyiceberg 직접 접속)
```

카탈로그 Postgres는 **CloudNativePG**가 관리하므로 서비스명에 `-rw`(쓰기)·`-ro`(읽기 전용)
접미사가 붙는다. 접미사 없는 `catalog-postgres`는 **존재하지 않는다**(`docs/conventions/k8s.md` §12).

## Trino는 쓰지 않는다

재설계에서 Trino는 제거 대상이고 ad-hoc 조회는 **Spark SQL**로 간다
(`docs/architectures/trino.md`). compose의 `trino`는 `--profile legacy-sql` 로만 뜬다.

## ⚠️ 보안 — 셀 출력

원천은 **비식별 연구 데이터셋이지만 DUA 대상**이다(`docs/security.md`).
셀 출력에 남은 행이 그대로 커밋되지 않도록 `nbstripout` pre-commit 훅이 걸려 있다.
훅을 우회해 강제 커밋하지 않는다.

## 1. 사전 점검 — port-forward가 살아 있는가

In [ ]:
import socket

# 접속 대상: (레이블, 포트, 필수 여부)
# 카탈로그 PG는 CloudNativePG가 만드는 `catalog-postgres-rw`로 port-forward한다.
ENDPOINTS = [
    ("spark-connect", 15002, True),
    ("catalog-postgres-rw", 15432, False),
    ("seaweedfs(s3)", 18333, False),
]

for label, port, required in ENDPOINTS:
    with socket.socket() as sock:
        sock.settimeout(1)
        alive = sock.connect_ex(("127.0.0.1", port)) == 0
    mark = "OK" if alive else ("MISSING" if required else "skip")
    print(f"[{mark:>7}] {label:<21} 127.0.0.1:{port}")

## 2. Spark Connect 접속

카탈로그 설정(JDBC URI·warehouse·S3 엔드포인트·자격증명)은 **서버 측**
`k8s/spark/spark-connect-server.yaml`에 이미 들어 있다.
→ 클라이언트는 주소만 알면 되고 **비밀정보를 노트북에 두지 않는다**.

In [ ]:
import os

from pyspark.sql import SparkSession

SPARK_REMOTE = os.environ.get("SPARK_REMOTE", "sc://localhost:15002")

spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
print(spark.version)

## 3. 카탈로그 탐색

카탈로그 이름은 **전 엔진 `iceberg`로 통일**한다. JDBC 카탈로그는 `catalog_name`으로
레지스트리를 분할하므로, 이름이 다르면 같은 DB를 봐도 서로의 테이블이 보이지 않는다.

In [ ]:
spark.sql("SHOW NAMESPACES IN iceberg").show()

In [ ]:
# 네임스페이스별 테이블 — 위 결과에 맞춰 바꿔 쓴다.
NAMESPACE = "poc"

spark.sql(f"SHOW TABLES IN iceberg.{NAMESPACE}").show()

## 4. 조회 → pandas

EDA는 pandas로 넘겨서 한다. **`toPandas()`는 드라이버 메모리에 전량 적재**하므로
반드시 집계하거나 `limit`을 건 뒤에 호출한다.

In [ ]:
TABLE = f"iceberg.{NAMESPACE}.sample"

df = spark.sql(f"SELECT * FROM {TABLE} LIMIT 100").toPandas()
df.head()

## 5. Iceberg 메타데이터

스냅샷·파일 목록은 메타데이터 테이블로 본다. small-files 누적을 확인할 때 쓴다
(유지보수 정책은 `docs/operations.md`).

In [ ]:
spark.sql(f"SELECT * FROM {TABLE}.snapshots ORDER BY committed_at DESC").show(
    5, truncate=False
)

## 6. (선택) pyiceberg 직접 접속

Spark를 거치지 않고 pyarrow로 바로 읽는 경로. **Dagster 적재 에셋과 같은 코드**를
쓰므로 적재 결과를 그대로 재현·검증할 수 있다.

접속 파라미터의 단일 출처는 `common.constants`이고, 대상 전환은 **`.env`로** 한다.
호스트에서 K8s 카탈로그를 보려면 `.env`에 아래가 채워져 있어야 한다
(키 목록은 `.env.example` 참고 — 값은 커밋하지 않는다):

```
ICEBERG_CATALOG_HOST=localhost
ICEBERG_CATALOG_PORT=15432
ICEBERG_CATALOG_DB=iceberg
ICEBERG_CATALOG_USER=iceberg
ICEBERG_CATALOG_PASSWORD=...      # k8s Secret catalog-pg-app 의 password
ICEBERG_S3_ENDPOINT=http://localhost:18333
ICEBERG_S3_ACCESS_KEY=...         # k8s Secret lakehouse-creds 의 s3-access-key
ICEBERG_S3_SECRET_KEY=...         # k8s Secret lakehouse-creds 의 s3-secret-key
```

시크릿이 **용도별로 나뉘어 있다** — PG 계정은 `catalog-pg-app`(CloudNativePG bootstrap 시크릿,
`basic-auth`라 키 이름이 `username`/`password`로 고정), S3 키는 `lakehouse-creds`다.
같은 비밀번호를 두 시크릿에 중복 보관하지 않는 것이 규약이다(`docs/conventions/k8s.md` §12).

🔴 **엔드포인트와 S3 키는 한 쌍이다.** 엔드포인트만 K8s로 바꾸고 키를 공용
`AWS_ACCESS_KEY_ID`(compose SeaweedFS용)로 두면 **네임스페이스·테이블 나열까지는
성공하고 `load_table`에서 `ACCESS_DENIED`** 로 죽는다(2026-08-19 실측).
부분 성공이라 원인을 오해하기 쉽다.
`ICEBERG_S3_*`를 비우면 공용 `AWS_*`로 폴백한다(compose 단독 구성용).

카탈로그 키가 미설정이면 기본값이 **compose**(`postgres:5432/iceberg_catalog`)를
가리켜 호스트에서는 이름 해석에 실패한다.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

# repo 루트의 .env를 읽어 os.environ에 주입한다(노트북 cwd = notebooks/).
load_dotenv(Path.cwd().parent / ".env")

from dagster_project.common.constants import (  # noqa: E402
    AWS_REGION,
    CATALOG_NAME,
    ICEBERG_CATALOG_URI,
    S3_ACCESS_KEY_ID,
    S3_ENDPOINT,
    S3_SECRET_ACCESS_KEY,
    WAREHOUSE,
)
from pyiceberg.catalog.sql import SqlCatalog  # noqa: E402  (env 주입 후 import)

catalog = SqlCatalog(
    CATALOG_NAME,
    **{
        "uri": ICEBERG_CATALOG_URI,
        "warehouse": WAREHOUSE,
        "s3.endpoint": S3_ENDPOINT,
        # ICEBERG_S3_* 가 있으면 그 값, 없으면 공용 AWS_* 로 폴백된 값이다.
        "s3.access-key-id": S3_ACCESS_KEY_ID,
        "s3.secret-access-key": S3_SECRET_ACCESS_KEY,
        "s3.region": AWS_REGION,
        "s3.path-style-access": "true",
    },
)
print(catalog.list_namespaces())

## 7. 정리

Spark Connect 서버는 **유일한 상주 컴퓨트**다. 오래 안 쓸 거면 세션을 닫고
`kubectl scale deploy/spark-connect --replicas=0` 으로 내린다.

In [ ]:
spark.stop()